<a href="https://colab.research.google.com/github/Man980/py_project/blob/main/fds_capstone_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FDS - CAPSTONE PROJECT

In [7]:
# Load libraries
import pandas as pd
import numpy as np
import networkx as nx
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import warnings
warnings.filterwarnings("ignore")


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [11]:
# ==============================================================================
# PHASE 0 : CHARGEMENT ET EXPLORATION
# ==============================================================================

path = "/content/drive/MyDrive/mobile_money_synthetic_dataset.csv"
df = pd.read_csv(path)

print("="* 70)
print("PHASE 0: EXPLORATION")
print("="* 70)
print("Dimentions : ", df.shape)
print("\nTypes de colonnes :\n", df.dtypes)
print("Valeurs manquantes :\n", df.isnull().sum())
print("Doublons sur ID : ", df["ID"].duplicated().sum())
print("Doublons sur toutes les colonnes :", df.duplicated().sum())

for col in ["account_type", "region", "transact_type", "day_of_week"]:
  print(f"\n--- Modalites de {col} ---")
  print(df[col].value_counts())

print("\n Statistiques Descriptives \n", df.describe())

Dimentions :  (1000000, 17)

Types de colonnes :
 ID                      object
Transact_amount        float64
cash_in                  int64
cash_out                 int64
hour                     int64
month                    int64
day_of_week             object
account_type            object
name_orig               object
name_dest               object
sender_bal_before      float64
sender_bal_after       float64
receiver_bal_before    float64
receiver_bal_after     float64
region                  object
transact_type           object
isFraud                  int64
dtype: object
Valeurs manquantes :
 ID                     0
Transact_amount        0
cash_in                0
cash_out               0
hour                   0
month                  0
day_of_week            0
account_type           0
name_orig              0
name_dest              0
sender_bal_before      0
sender_bal_after       0
receiver_bal_before    0
receiver_bal_after     0
region                 0
transact_typ

In [4]:
df.head(20)

,ID,Transact_amount,cash_in,cash_out,hour,month,day_of_week,account_type,name_orig,name_dest,sender_bal_before,sender_bal_after,receiver_bal_before,receiver_bal_after,region,transact_type,isFraud
0,TX_00000000,1418.09,0,1,18,6,Monday,Agent,ACC_0036301,ACC_0047083,22246.67,20828.58,9296.65,10714.74,Cite Soleil,CASH_OUT,0
1,TX_00000001,17918.34,0,1,19,8,Saturday,Personnel,ACC_0018768,ACC_0026943,17918.34,0.00,88933.70,106852.04,Cite Soleil,CASH_OUT,0
2,TX_00000002,502.02,1,0,0,7,Friday,Personnel,ACC_0067838,ACC_0026626,20292.78,19790.76,8945.22,9447.24,Port-au-Prince,CASH_IN,0
3,TX_00000003,1667.36,0,1,15,6,Thursday,Personnel,ACC_0025768,ACC_0053596,6753.78,5086.42,26292.21,27959.57,Petion-Ville,CASH_OUT,0
4,TX_00000004,4316.41,1,0,13,10,Wednesday,Agent,ACC_0083894,ACC_0079307,11291.70,6975.29,24622.83,28939.24,Port-au-Prince,CASH_IN,0
5,TX_00000005,1147.14,0,1,8,7,Friday,Personnel,ACC_0059526,ACC_0084689,11290.27,10143.13,30444.17,31591.31,Port-au-Prince,CASH_OUT,0
6,TX_00000006,2177.66,0,1,9,9,Sunday,Personnel,ACC_0056757,ACC_0071001,31234.45,29056.79,21667.55,23845.21,Port-au-Prince,CASH_OUT,0
7,TX_00000007,1181.36,0,0,18,6,Monday,Personnel,ACC_0020794,ACC_0099891,29280.50,28099.14,12165.33,13346.69,Petion-Ville,PAYMENT,0
8,TX_00000008,5582.80,0,0,17,6,Wednesday,Personnel,ACC_0093154,ACC_0060417,25710.91,20128.11,226246.25,231829.05,Port-au-Prince,TRANSFER,0
9,TX_00000009,732.04,1,0,15,12,Saturday,Personnel,ACC_0083526,ACC_0063916,6030.07,5298.03,28060.03,28792.07,Petion-Ville,CASH_IN,0


In [ ]:
df["name_orig"].value_counts()

In [ ]:
df[["transact_type", "cash_in", "cash_out"]].value_counts()

In [ ]:
df.head(50)

In [6]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 17 columns):
 #   Column               Non-Null Count    Dtype  
---  ------               --------------    -----  
 0   ID                   1000000 non-null  object 
 1   Transact_amount      1000000 non-null  float64
 2   cash_in              1000000 non-null  int64  
 3   cash_out             1000000 non-null  int64  
 4   hour                 1000000 non-null  int64  
 5   month                1000000 non-null  int64  
 6   day_of_week          1000000 non-null  object 
 7   account_type         1000000 non-null  object 
 8   name_orig            1000000 non-null  object 
 9   name_dest            1000000 non-null  object 
 10  sender_bal_before    1000000 non-null  float64
 11  sender_bal_after     1000000 non-null  float64
 12  receiver_bal_before  1000000 non-null  float64
 13  receiver_bal_after   1000000 non-null  float64
 14  region               1000000 non-null  object 
 15 

In [ ]:
# ==================================================
# Générateur Dataset Mobile Money Synthétique
# ==================================================

import numpy as np
import pandas as pd


# ==================================================
# Paramètres
# ==================================================

np.random.seed(42)

N_TRANSACTIONS = 1_000_000
N_ACCOUNTS = 100_000



# ==================================================
# 1. Génération des comptes
# ==================================================

accounts = pd.DataFrame({

    "account_id": [
        "ACC_" + str(i).zfill(7)
        for i in range(N_ACCOUNTS)
    ]

})


# -------------------------------
# Type de compte
# -------------------------------

account_types = [
    "Personnel",
    "Marchand",
    "Agent",
    "Entreprise"
]


accounts["account_type"] = np.random.choice(
    account_types,
    size=N_ACCOUNTS,
    p=[
        0.75,
        0.15,
        0.08,
        0.02
    ]
)



# -------------------------------
# Région
# -------------------------------

regions = [
    "Port-au-Prince",
    "Delmas",
    "Petion-Ville",
    "Kenscoff",
    "Cite Soleil",
    "Tabarre",
    "Carrefour",
    "Gressier"
]


prob_region = [
    0.45,
    0.15,
    0.10,
    0.08,
    0.07,
    0.05,
    0.06,
    0.04
]


accounts["region"] = np.random.choice(
    regions,
    size=N_ACCOUNTS,
    p=prob_region
)



# ==================================================
# 2. Génération des transactions
# ==================================================

df = pd.DataFrame()


# Identifiant transaction

df["ID"] = [
    "TX_" + str(i).zfill(8)
    for i in range(N_TRANSACTIONS)
]



# Type de transaction

transaction_types = [
    "CASH_IN",
    "CASH_OUT",
    "TRANSFER",
    "PAYMENT"
]


df["transact_type"] = np.random.choice(
    transaction_types,
    size=N_TRANSACTIONS,
    p=[
        0.25,
        0.25,
        0.35,
        0.15
    ]
)



# ==================================================
# 3. Comptes origine et destination
# ==================================================

df["name_orig"] = np.random.choice(
    accounts["account_id"],
    size=N_TRANSACTIONS
)


df["name_dest"] = np.random.choice(
    accounts["account_id"],
    size=N_TRANSACTIONS
)


# Eviter les transactions vers soi-même

mask = df["name_orig"] == df["name_dest"]

df.loc[mask, "name_dest"] = np.random.choice(
    accounts["account_id"],
    size=mask.sum()
)



# ==================================================
# 4. Variables temporelles
# ==================================================

dates = (
    pd.Timestamp("2025-01-01")
    +
    pd.to_timedelta(
        np.random.randint(
            0,
            365*24,
            N_TRANSACTIONS
        ),
        unit="h"
    )
)


df["hour"] = dates.hour

df["month"] = dates.month

df["day_of_week"] = dates.day_name()



# ==================================================
# 5. Montants des transactions
# ==================================================

df["Transact_amount"] = np.random.lognormal(
    mean=8,
    sigma=1.2,
    size=N_TRANSACTIONS
)


df["Transact_amount"] = (
    df["Transact_amount"]
    .round(2)
)



# ==================================================
# 6. Soldes avant transaction
# ==================================================

df["sender_bal_before"] = (
    np.random.lognormal(
        mean=10,
        sigma=1,
        size=N_TRANSACTIONS
    )
    .round(2)
)



df["receiver_bal_before"] = (
    np.random.lognormal(
        mean=10,
        sigma=1,
        size=N_TRANSACTIONS
    )
    .round(2)
)



# Empêcher un retrait supérieur au solde

df["Transact_amount"] = np.minimum(
    df["Transact_amount"],
    df["sender_bal_before"]
)



# Solde après transaction

df["sender_bal_after"] = (
    df["sender_bal_before"]
    -
    df["Transact_amount"]
)



df["receiver_bal_after"] = (
    df["receiver_bal_before"]
    +
    df["Transact_amount"]
)



# ==================================================
# 7. Variables CASH
# ==================================================

df["cash_in"] = (
    df["transact_type"]
    ==
    "CASH_IN"
).astype(int)



df["cash_out"] = (
    df["transact_type"]
    ==
    "CASH_OUT"
).astype(int)



# ==================================================
# 8. Informations comptes
# ==================================================

account_info = accounts.set_index(
    "account_id"
)


df["account_type"] = (
    df["name_orig"]
    .map(account_info["account_type"])
)


df["region"] = (
    df["name_orig"]
    .map(account_info["region"])
)



# ==================================================
# 9. Création du label fraude
# ==================================================

df["isFraud"] = 0



# Cas 1 : transactions exceptionnellement élevées

threshold = df["Transact_amount"].quantile(0.995)


df.loc[
    df["Transact_amount"] > threshold,
    "isFraud"
] = 1



# Cas 2 : grosses transactions la nuit

df.loc[
    (df["hour"] <= 4)
    &
    (df["Transact_amount"] > 50000),
    "isFraud"
] = 1



# Cas 3 : comportement agent suspect

agent_accounts = accounts[
    accounts["account_type"] == "Agent"
]["account_id"]


df.loc[
    (
        df["name_orig"]
        .isin(agent_accounts)
    )
    &
    (
        df["Transact_amount"] > 100000
    ),
    "isFraud"
] = 1



# ==================================================
# 10. Dataset final
# ==================================================

dataset = df[
[
    "ID",
    "Transact_amount",
    "cash_in",
    "cash_out",
    "hour",
    "month",
    "day_of_week",
    "account_type",
    "name_orig",
    "name_dest",
    "sender_bal_before",
    "sender_bal_after",
    "receiver_bal_before",
    "receiver_bal_after",
    "region",
    "transact_type",
    "isFraud"
]
]



# ==================================================
# 11. Export CSV
# ==================================================

path = "/content/drive/MyDrive/mobile_money_synthetic_dataset.csv"


dataset.to_csv(
    path,
    index=False
)



# ==================================================
# Vérifications
# ==================================================

print(dataset.head())

print("\nDimension du dataset :")
print(dataset.shape)


print("\nRépartition fraude :")
print(dataset["isFraud"].value_counts())


print("\nTaux fraude :")
print(round(dataset["isFraud"].mean()*100,2), "%")


print("\nFichier sauvegardé :", path)